# Akkadian Translation Inference

Single-model inference using the `akkadian` package.
Beam search with MBR ChrF reranking.

## Setup

In [1]:
import os

ON_KAGGLE = os.path.exists("/kaggle")

if ON_KAGGLE:
    PKG_DIR = "/kaggle/input/datasets/ebinan92/akkadian-v2-pkg"
    os.system(
        f"pip install --no-deps --no-index --find-links={PKG_DIR}"
        " portalocker regex tabulate colorama"
        " sacrebleu polars polars-runtime-32"
        " ctranslate2"
        " akkadian-v2"
    )

Looking in links: /kaggle/input/datasets/ebinan92/akkadian-v2-pkg
Processing /kaggle/input/datasets/ebinan92/akkadian-v2-pkg/portalocker-3.2.0-py3-none-any.whl
Processing /kaggle/input/datasets/ebinan92/akkadian-v2-pkg/sacrebleu-2.6.0-py3-none-any.whl
Processing /kaggle/input/datasets/ebinan92/akkadian-v2-pkg/polars_runtime_32-1.39.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/ebinan92/akkadian-v2-pkg/ctranslate2-4.7.1-cp312-cp312-manylinux_2_35_x86_64.whl
Processing /kaggle/input/datasets/ebinan92/akkadian-v2-pkg/akkadian_v2-0.1.0-py3-none-any.whl


In [2]:
from akkadian.inference.infer import InferenceArgs, run_inference

## Configuration

Set model path, input/output paths, and generation parameters.

In [3]:
if ON_KAGGLE:
    MODEL_PATH = "/kaggle/input/models/shelterw/deeppast/transformers/byt5-large-akkadian/1"
    CT2_MODEL_PATH = "/kaggle/input/models/ebinan92/byt5-xl-new-data-averaged_ct2_float32/pytorch/default/1"
    INPUT_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    OUTPUT_PATH = "submission.csv"
else:
    MODEL_PATH = "./outputs/finetune/best_model/"
    CT2_MODEL_PATH = None
    INPUT_PATH = "./input/deep-past-initiative-machine-translation/test.csv"
    OUTPUT_PATH = "./outputs/predictions.csv"

In [4]:
args = InferenceArgs(
    model=MODEL_PATH,
    ct2_model_path=CT2_MODEL_PATH,
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    per_device_eval_batch_size=2,
    generation_strategy="beam",
    generation_num_candidates=12,
    rerank_strategy="first",
    ct2_quantization="float32",
    ct2_tensor_parallel=ON_KAGGLE,
    ct2_tmp_dir="/kaggle/working" if ON_KAGGLE else None,
    gpu_ids="0,1" if ON_KAGGLE else None,
    max_length=3096,
    context_num_prev=2,
    context_max_bytes=1024,
    prepend_context=True
)
args

InferenceArgs(model='/kaggle/input/models/shelterw/deeppast/transformers/byt5-large-akkadian/1', input_path='/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv', output_path='submission.csv', sample_scores_path='./outputs/eval_sample_scores.csv', input_transliteration_column='transliteration', output_id_column='id', output_confidence_column='confidence', add_transliteration=False, apply_transliteration_char_normalization=True, compute_confidence=False, dryrun=False, fold=None, max_length=3096, per_device_eval_batch_size=2, generation_strategy='beam', generation_num_candidates=12, generation_max_new_tokens=1024, generation_max_new_tokens_input_ratio=1.7, generation_length_penalty=1.3, generation_early_stopping=True, no_repeat_ngram_size=None, generation_temperature=0.6, generation_top_k=0, generation_top_p=1.0, rerank_strategy='first', gpu_ids='0,1', prepend_context=True, context_num_prev=2, context_max_bytes=1024, ct2_model_path='/kaggle/input/models/ebinan92/

## Run Inference

In [5]:
output_df = run_inference(args)
output_df.head(10)

Context prepended: 3/4 rows (num_prev=2, max_bytes=1024)
Launching CT2 tensor parallel on 2 GPUs...


id,translation
i64,str
0,"""From the Kanesh colony to the …"
1,"""According to the tablet in the…"
2,"""When you hear our tablet, writ…"
3,"""I have sent the copies of our …"


In [6]:
import polars as pl

pred_col = "prediction" if "prediction" in output_df.columns else "translation"
id_col = next(c for c in ["id", "oare_id", "sample_id"] if c in output_df.columns)
submission_df = output_df.select(
    pl.col(id_col).alias("id"),
    pl.col(pred_col).alias("translation"),
)

submission_path = "submission.csv" if ON_KAGGLE else OUTPUT_PATH
submission_df.write_csv(submission_path)
print(f"Submission saved to {submission_path}: {len(submission_df)} rows")
submission_df.head()

Submission saved to submission.csv: 4 rows


id,translation
i64,str
0,"""From the Kanesh colony to the …"
1,"""According to the tablet in the…"
2,"""When you hear our tablet, writ…"
3,"""I have sent the copies of our …"


## Validation (optional)

Run on training data with fold split to evaluate model performance.

In [7]:
# if ON_KAGGLE:
#     EVAL_INPUT_PATH = "/kaggle/input/datasets/ebinan92/akkadian-v2-pkg/train_with_akt_splited.csv"
# else:
#     EVAL_INPUT_PATH = "./custom_input/train_with_akt_splited.csv"
#
# eval_args = InferenceArgs(
#     model=MODEL_PATH,
#     input_path=EVAL_INPUT_PATH,
#     output_path="./outputs/eval_predictions.csv",
#     sample_scores_path="./outputs/eval_sample_scores.csv",
#     fold=0,
#     per_device_eval_batch_size=2,
#     generation_num_beams=8,
#     max_length=1024,
# )
# eval_df = run_inference(eval_args)